# 03 · Retraining & Rollback — *When to refresh the model, and how to undo a bad release*

**MLOps Monitoring · Coaching Session · Module 3 (Machine Learning & GenAI)**

---

Notebook 02 showed our 2021 model is now about **30% off** and always too low. The obvious fix is "retrain it". But:

1. **How do we test** a new model honestly?
2. **When** should we retrain — every month? Only when something breaks?
3. **Which data** do we train on — everything since 2017, or only recent years?
4. **How do we decide** the new model is really better before it goes live?
5. **What if the new model is worse** after it goes live?

> **Analogy — servicing your car 🚗**
> * **Scheduled:** service every 10,000 km, even if nothing feels wrong.
> * **Triggered:** the engine warning light comes on → go now.
> * **Test drive** before you take the car back (validation).
> * **Keep the old parts** until you're sure the new ones work (rollback).

## 0 · Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from mlops_utils import (load_hdb, make_model, score, register_model, set_champion, load_champion,
                         show_registry, reset_registry, FEATURES, TARGET, DEPLOY_DATE)

pd.set_option("display.float_format", "{:,.2f}".format)

df = load_hdb()
reference  = df[df.date <  DEPLOY_DATE]
production = df[df.date >= DEPLOY_DATE]
print(f"Data from {df.month.min()} to {df.month.max()}")

---
## Part A · Test on the *most recent* months, not a random sample 🎲

In L03 we used a **random** train/test split. For data that changes over time, that is **too kind**.

> **Analogy — the mock exam 📝** If your mock exam mixes questions from *this year's* paper into your practice set, you will score well in the mock and then struggle in the real exam.
> A random split lets the model **peek at the future**: it trains on some Dec 2021 deals and is tested on other Dec 2021 deals.

The honest test: **train on older months, test on the newest months.** That is what the model will face in real life.

Let's see which test gave the more honest forecast of how v1 *really* did in its first 6 months live.

In [ ]:
# 1) Random split (the L03 way)
train_r, test_r = train_test_split(reference, test_size=0.2, random_state=42)
m_random = make_model().fit(train_r[FEATURES], train_r[TARGET])
random_split = score(test_r[TARGET], m_random.predict(test_r[FEATURES]))

# 2) Time split: train up to Jun 2021, test on the latest 6 months (Jul–Dec 2021)
# 🔧 YOUR TURN: the TEST set is everything ON or AFTER the cut-off date
CUT = "2021-07-01"
train_t = reference[reference.date <  CUT]
test_t  = reference[reference.date ____ CUT]
m_time = make_model().fit(train_t[FEATURES], train_t[TARGET])
time_split = score(test_t[TARGET], m_time.predict(test_t[FEATURES]))

# 3) What REALLY happened: model trained on all 2017–2021, used in Jan–Jun 2022
m_v1 = make_model().fit(reference[FEATURES], reference[TARGET])
first_6m = production[production.date < "2022-07-01"]
reality = score(first_6m[TARGET], m_v1.predict(first_6m[FEATURES]))

pd.DataFrame({"Random split (L03 way)": random_split,
              "Time split (latest 6 months)": time_split,
              "REALITY: first 6 months live": reality}).T[["MAPE_%", "bias"]]

<details><summary>▸ Hint</summary>

The test set is the *later* part: `>=`.
</details>

**What do you notice?** The random split promises a much smaller error than what really happened. The **time split is closer to reality**, and it even shows the negative bias. ✅ **Always validate on the most recent months.**

---
## Part B · When to retrain — write the "warning light" 🚨

Two styles, often combined:

| Style | Analogy | Good for | Weakness |
|---|---|---|---|
| **Scheduled** (e.g. every quarter) | Car service every 10,000 km | Simple, predictable | May retrain when not needed, or too late after a sudden shock |
| **Triggered** (when a check fails) | Engine warning light | Reacts to sudden change | Needs good monitoring and clear thresholds |

A simple trigger uses what we built in notebook 02: **MAPE got much worse than at launch**, or **bias is too large**.

In [ ]:
LAUNCH_MAPE = time_split["MAPE_%"]        # our honest estimate on launch day

def should_retrain(month_rows, model):
    s = score(month_rows[TARGET], model.predict(month_rows[FEATURES]))
    bias_pct = s["bias"] / month_rows[TARGET].mean() * 100
    reasons = []
    # 🔧 YOUR TURN: alarm if this month's MAPE is more than 1.5x the launch MAPE
    if s["MAPE_%"] > ____ * LAUNCH_MAPE:
        reasons.append(f"MAPE {s['MAPE_%']:.1f}% > 1.5 x launch ({LAUNCH_MAPE:.1f}%)")
    if abs(bias_pct) > 5:
        reasons.append(f"bias {bias_pct:+.1f}% (more than ±5%)")
    return reasons

for month, rows in production.groupby("month"):
    reasons = should_retrain(rows, m_v1)
    if reasons:
        print(f"🚨 First alarm in {month}: " + "; ".join(reasons))
        break

The warning light comes on in the **very first month** after launch, because of the bias rule. (The MAPE rule fires a few months later.) In a real system this would open a ticket or send a Slack message to the team.

---
## Part C · Which data to retrain on? Simulate 5 years of decisions ⏪

We **replay history**: start in Jan 2022 and walk forward one quarter at a time, as if we were living through it.

| Strategy | What it does | Analogy |
|---|---|---|
| **Never retrain** | Keep v1 forever | Using a 2021 street directory in 2026 |
| **Yearly, expanding window** | Every January, retrain on *all* data since 2017 | Studying for the exam with *every* past paper since 2017 |
| **Quarterly, sliding window** | Every quarter, retrain on only the *last N years* | Studying with the *recent* papers — the format changed |

This walk-forward replay is also called **backtesting** (you met it in L06 for time series).

⏳ This cell trains about 25 models and takes **1–2 minutes**.

In [ ]:
# 🔧 YOUR TURN: how many years of recent data should the sliding window keep?
WINDOW_YEARS = ____

quarters = pd.period_range(production.date.min(), production.date.max(), freq="Q")
results = []
models = {"Never retrain": m_v1, "Yearly, expanding": None, f"Quarterly, sliding {WINDOW_YEARS}y": None}

for q in quarters:
    start = q.start_time
    test = df[df.date.dt.to_period("Q") == q]
    if q.quarter == 1:                                            # every January
        train = df[df.date < start]
        models["Yearly, expanding"] = make_model().fit(train[FEATURES], train[TARGET])
    train = df[(df.date < start) & (df.date >= start - pd.DateOffset(years=WINDOW_YEARS))]
    models[f"Quarterly, sliding {WINDOW_YEARS}y"] = make_model().fit(train[FEATURES], train[TARGET])
    for name, m in models.items():
        results.append({"quarter": str(q), "strategy": name,
                        "MAPE": score(test[TARGET], m.predict(test[FEATURES]))["MAPE_%"]})

backtest = pd.DataFrame(results).pivot(index="quarter", columns="strategy", values="MAPE")
ax = backtest.plot(figsize=(12, 4), marker="o")
ax.set_ylabel("MAPE %"); ax.set_title("Backtest 2022–2026: error of each retraining strategy")
plt.xticks(rotation=45); plt.show()
backtest.mean().sort_values().to_frame("average MAPE %")

<details><summary>▸ Hint</summary>

Start with `2`. Then re-run with `1` and `4` and compare the average MAPE.
</details>

**What do you notice?**
* **Never retrain** gets worse and worse.
* **Yearly expanding** has a **saw-tooth**: better each January, then worse through the year. It also stays fairly high, because 2017 prices still "pull it down".
* **Quarterly sliding** stays low: it **forgets old prices that are no longer true**.

**Trade-off of a short window:** fewer rows, and rare flat types (e.g. `3Gen`, `MULTI-GENERATION`) may have very few examples. Try `WINDOW_YEARS = 1` and look at it.

---
## Part D · Champion vs challenger — the test drive 🏁

We never swap the live model just because we *trained* a new one. The new model (**challenger**) must **beat** the live one (**champion**) on the **most recent months**, which neither has trained on.

> **Analogy — the football team ⚽** The current goalkeeper (champion) keeps the spot until the new one (challenger) does better **in the same recent matches** — not in training.

**Our promotion rule (the "gate"):**
1. Challenger's MAPE is lower than champion's, **and**
2. No flat type gets more than 2 percentage points worse (no hidden losers).

In [ ]:
latest_month = df.date.max()
holdout = df[df.date > latest_month - pd.DateOffset(months=3)]            # newest 3 months: test only
train_window = df[(df.date <= latest_month - pd.DateOffset(months=3)) &
                  (df.date >  latest_month - pd.DateOffset(months=3, years=2))]

challenger = make_model().fit(train_window[FEATURES], train_window[TARGET])

def compare(champion, challenger, data):
    out = data[["flat_type", TARGET]].copy()
    for name, m in [("champion", champion), ("challenger", challenger)]:
        out[name] = (m.predict(data[FEATURES]) - data[TARGET]).abs() / data[TARGET] * 100
    by_type = out.groupby("flat_type")[["champion", "challenger"]].mean()
    by_type["change"] = by_type["challenger"] - by_type["champion"]
    overall = out[["champion", "challenger"]].mean()
    return overall, by_type

overall, by_type = compare(m_v1, challenger, holdout)
print(f"Test months: {holdout.month.min()} to {holdout.month.max()}  ({len(holdout):,} deals)")
print(f"Overall MAPE — champion v1: {overall['champion']:.1f}%   challenger: {overall['challenger']:.1f}%\n")
by_type

In [ ]:
# 🔧 YOUR TURN: complete the gate
passes_overall  = overall["challenger"] ____ overall["champion"]
passes_segments = (by_type["change"] <= 2).all()

if passes_overall and passes_segments:
    print("✅ Challenger passes the gate -> promote it as v2")
    register_model(challenger, "v2", notes="sliding 2y window, passed gate",
                   trained_until=str(train_window.month.max()),
                   metrics={"holdout_MAPE": overall["challenger"]})
    set_champion("v2")
else:
    print("❌ Challenger rejected — champion stays live")
show_registry()

<details><summary>▸ Hint</summary>

The challenger passes when its MAPE is *lower*: `<`.
</details>

💡 **In practice**, after the challenger passes, many teams retrain it once more *including* the 3 test months, so it has the very latest prices.

---
## Part E · Rollback — undo a bad release ⏮️

Things go wrong even with good process. Here is a real-world style accident:

> A data engineer changes the price pipeline. For three months, prices are stored **in thousands** (`650` instead of `650000`). Nobody notices. A "quick hotfix" model **v3** is trained on that data and pushed live **without** going through the gate.

> **Analogy — the phone update 📱** Your phone installs an update and your banking app stops working. You want a button that says **"go back to the previous version"**. That only works if the old version was **kept** and you know **which** one it was.

In [ ]:
# The accident: prices in thousands for 3 months of training data
bad_train = train_window.copy()
bug_months = sorted(bad_train.month.unique())[-3:]
bad_train.loc[bad_train.month.isin(bug_months), TARGET] /= 1000
print("Months with the bug:", bug_months)

v3 = make_model().fit(bad_train[FEATURES], bad_train[TARGET])
register_model(v3, "v3", notes="hotfix — SKIPPED the gate", trained_until=str(bad_train.month.max()))
set_champion("v3")                              # pushed live without checks 😬
show_registry()

### Monitoring catches it on the next batch of real prices

In [ ]:
live_version, live_model = load_champion()
s = score(holdout[TARGET], live_model.predict(holdout[FEATURES]))
print(f"Live model {live_version}: MAPE = {s['MAPE_%']:.1f}%   bias = S${s['bias']:,.0f}")
if s["MAPE_%"] > 1.5 * LAUNCH_MAPE:
    print("🚨 ALARM — error far above normal. Roll back first, investigate second.")

In [ ]:
# 🔧 YOUR TURN: roll back to the last GOOD version
set_champion(____)

live_version, live_model = load_champion()
s = score(holdout[TARGET], live_model.predict(holdout[FEATURES]))
print(f"Live model is now {live_version}: MAPE = {s['MAPE_%']:.1f}% ✅")
show_registry()

<details><summary>▸ Hint</summary>

The last version that passed the gate was `"v2"`.
</details>

**Rollback took one line** because we did three things in advance:
1. **Kept every version** (never overwrite the old model file).
2. **Recorded which version is live** (the "champion" pointer).
3. **Wrote down how each was trained** (notes, dates, metrics).

Tools like **MLflow's Model Registry** do exactly this for big teams (see optional notebook 04). The idea is the same as our small `registry.json`.

**Order of actions in an incident:** ① roll back → ② tell users → ③ find the root cause (the pipeline bug) → ④ add a check so it can't happen again (e.g. "alarm if the median price drops more than 50% month on month").

---
## Part F · 🔧 YOUR TURN — write your group's retraining policy

Fill in this table for your project (double-click to edit). This is a great slide for your final presentation.

| Question | Our answer |
|---|---|
| **Trigger:** scheduled, triggered, or both? Which thresholds? | *e.g. quarterly, plus alarm if MAPE > 1.5× launch or bias beyond ±5%* |
| **Window:** expanding or sliding? How many years? | |
| **Validation:** which months are held out for testing? | *e.g. the latest 3 months* |
| **Gate:** what must the challenger beat? | |
| **Rollback:** how do we switch back, and who decides? | |
| **New categories:** what happens when a new town / flat model appears? | |

---
### Key takeaways
* **Test on the latest months**, not a random split — random splits are too optimistic when the world changes.
* **Retrain on a schedule *and* on alarms.**
* In a fast-moving market, a **sliding window** usually beats "use all history".
* **Champion vs challenger:** a new model goes live only if it wins on recent unseen data — overall *and* per segment.
* **Keep every version + a "live" pointer** → rollback is one line.